### Imports


In [ ]:
# !pip install google-genai scipy -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import (
    confusion_matrix,
    roc_curve,
    auc,
    precision_recall_fscore_support,
)
import re
import math
import ast
import os
import sys
import importlib.util
from collections import Counter
from pathlib import Path
import builtins
import signal
import subprocess
import tempfile

from scipy.stats import binomtest, norm

import multiprocessing
import traceback

from typing import List, Dict, Any, Tuple, Callable, Optional, Union

### Constants


In [ ]:
MODEL_NAME = "gemini"
MODEL_PATH = "gemini-2.5-flash"

# Project root directory (absolute path to avoid confusion with relative paths)
BASE_DIR = "../../../"  # or '.'

# Derived paths
DATASET_PATH = f"../../../datasets/core/sanitized-mbpp-sample-100.json"

# Watermark parameters
Z_THRESHOLD = 2.12  # corresponds to a p-value of approximately 0.017 and confidence level of 98.3%
P_THRESHOLD = norm.sf(Z_THRESHOLD)
# GAMMA = 0.514161
SEED_KEY = "exp2025"
SMALL_SAMPLE_THRESHOLD = 30
COMMENT_ENABLED = True
ITER_CAP = 5

In [ ]:
df = pd.read_json(DATASET_PATH, lines=True)
df = df[:3]  # For testing purposes, limit to first 2 records
# df = df[df['task_id'].isin(missing_ids)]
df.shape

In [ ]:
# Experiment parameters
EXPERIMENT_NUMBER = "exp5"
EXP_VERSION = "v1"
GENERATION_TYPE = "during"  # 'during' or 'after'
SAMPLE_SIZE = df.shape[0]
DATASET = "mbpp"

RESULTS_CSV = f"{BASE_DIR}/results/raw/{MODEL_NAME}_{EXPERIMENT_NUMBER}_{GENERATION_TYPE}_gen_{EXP_VERSION}_{SAMPLE_SIZE}_{DATASET}.csv"
OUTPUT_DIR = f"{BASE_DIR}/output/{MODEL_NAME}_{EXPERIMENT_NUMBER}_{GENERATION_TYPE}_gen_{EXP_VERSION}_{SAMPLE_SIZE}_{DATASET}"

### Adaptive Green Tokens


In [ ]:
import hashlib
import random

## MBPP letter frequency distribution

# freq = {
#  'a': 23, 'b': 9, 'c': 24, 'd': 11, 'e': 28, 'f': 17,
#  'g': 5, 'h': 2, 'i': 43, 'j': 6, 'k': 3, 'l': 32,
#  'm': 27, 'n': 37, 'o': 4, 'p': 14, 'q': 1, 'r': 44,
#  's': 56, 't': 41, 'u': 2, 'v': 5, 'w': 4, 'x': 18,
#  'y': 2, 'z': 1
# }

freq_data = {
    "letter_freqs": {
        "a": 110,
        "b": 44,
        "c": 112,
        "d": 52,
        "e": 71,
        "f": 60,
        "g": 25,
        "h": 29,
        "i": 183,
        "j": 34,
        "k": 25,
        "l": 128,
        "m": 119,
        "n": 209,
        "o": 17,
        "p": 46,
        "q": 2,
        "r": 162,
        "s": 190,
        "t": 160,
        "u": 7,
        "v": 20,
        "w": 12,
        "x": 67,
        "y": 16,
        "z": 5,
    },
    "total_identifiers": 1905,
}


def calculate_gamma(
    letter_counts: Dict[str, int], total_count: int, green_letters: set
) -> float:
    if total_count == 0:
        return 0.0

    gamma = 0.0
    for letter in green_letters:
        if letter in letter_counts:
            p_letter = letter_counts[letter] / total_count
            gamma += p_letter

    return gamma


def get_dynamic_tier_counts(
    G, task_id, const=SEED_KEY, wH_range=(0.40, 0.60), wM_range=(0.25, 0.40)
) -> Dict[str, float]:
    #! deterministic seed
    seed_val = int(hashlib.md5((const).encode()).hexdigest(), 16)
    rnd = random.Random(seed_val)  # local RNG so no global side effects

    # sample weights uniformly within ranges
    wH = rnd.uniform(*wH_range)
    wM = rnd.uniform(*wM_range)

    # ensure wL within bounds by clipping if necessary
    wL = 1.0 - wH - wM
    # if wL out of [0.10, 0.35], nudge wM to make it valid
    if wL < 0.10:
        # reduce wH or wM to lift wL
        deficit = 0.10 - wL
        # reduce wH first, then wM
        wH = max(wH_range[0], wH - deficit * 0.6)
        wM = max(wM_range[0], wM - deficit * 0.4)
        wL = 1.0 - wH - wM
    elif wL > 0.35:
        excess = wL - 0.35
        wH = min(wH_range[1], wH + excess * 0.6)
        wM = min(wM_range[1], wM + excess * 0.4)
        wL = 1.0 - wH - wM

    # integer counts
    gH = max(1, int(math.floor(G * wH)))
    gM = max(1, int(math.floor(G * wM)))
    gL = G - gH - gM
    # fix rounding issues
    if gL < 1:
        # push some from gH/gM to gL
        if gM > 1:
            gM -= 1
            gL += 1
        else:
            gH -= 1
            gL += 1

    return {"wH": wH, "wM": wM, "wL": wL, "gH": gH, "gM": gM, "gL": gL}


def get_red_green_sets(
    task_id, const=SEED_KEY, freq_dict=freq_data["letter_freqs"]
) -> Tuple[set, set, float]:

    task_id = str(task_id)

    # ----- Step 1: sort by frequency -----
    sorted_letters = sorted(freq_dict.keys(), key=lambda k: freq_dict[k], reverse=True)
    n = len(sorted_letters)
    high = sorted_letters[:8]
    medium = sorted_letters[8:16]
    low = sorted_letters[16:]

    # print("HIGH TIER:", high)
    # print("MEDIUM TIER:", medium)
    # print("LOW TIER:", low)

    #! ----- Step 2: unique seed -----
    seed_val = int(hashlib.md5((const).encode()).hexdigest(), 16)
    random.seed(seed_val)

    # print("RANDOM SEED:", seed_val)

    # ----- Step 3: adaptive ratio -----
    ratio = 0.5 + (random.random() - 0.5) * 0.2  # between 0.4–0.6
    green_count = int(n * ratio)

    # print(f"GREEN RATIO: {ratio:.3f}, GREEN COUNT: {green_count}/{n}")

    # ----- Step 4: weighted selection -----
    counts = get_dynamic_tier_counts(green_count, task_id, const="exp2025")
    num_high = counts["gH"]
    num_medium = counts["gM"]
    num_low = counts["gL"]
    # num_high = int(green_count * 0.5)
    # num_medium = int(green_count * 0.3)
    # num_low = green_count - num_high - num_medium

    # print("HIGH COUNT:", num_high)
    # print("MEDIUM COUNT:", num_medium)
    # print("LOW COUNT:", num_low)

    green_letters = (
        random.sample(high, min(num_high, len(high)))
        + random.sample(medium, min(num_medium, len(medium)))
        + random.sample(low, min(num_low, len(low)))
    )

    # print("INITIAL GREEN LETTERS:", green_letters)

    # Fill up if short (due to small tiers)
    while len(green_letters) < green_count:
        extra = random.choice(sorted_letters)
        if extra not in green_letters:
            green_letters.append(extra)

    # Remaining letters → red set
    red_letters = [l for l in sorted_letters if l not in green_letters]
    return set(green_letters), set(red_letters), ratio

### Helper Methods


In [ ]:
import ast
import os
import sys
import importlib.util
from collections import Counter
from pathlib import Path
import builtins
import keyword

COMMON_STD_METHODS = {
    "self",
    "re",
    "cls",
    "append",
    "join",
    "dummy_function",
    "find",
    "kwargs",
    "args",
    "range",
    "print",
    "len",
    "dict",
    "list",
    "str",
    "int",
    "float",
    "set",
    "tuple",
    "os",
    "np",
    "subprocess",
    "now",
    "today",
    "timedelta",
    "strptime",
    "date",
    "time",
    "datetime",
    "logging",
    "log",
    "info",
    "debug",
    "error",
    "warning",
    "exception",
    "lower",
    "upper",
    "strip",
    "split",
    "replace",
    "endswith",
    "startswith",
    "append",
    "extend",
    "insert",
    "remove",
    "pop",
    "sort",
    "clear",
    "keys",
    "values",
    "items",
    "get",
    "update",
    "copy",
    "format",
    "join",
    "count",
    "index",
}

BUILTIN_NAMES = set(dir(builtins)).union(COMMON_STD_METHODS)


class CodeNavigator(ast.NodeVisitor):
    def __init__(self):
        self.public_classes = set()
        self.non_public_classes = set()
        self.public_funcs = set()
        self.non_public_funcs = set()
        self.public_vars = set()
        self.non_public_vars = set()
        self.docstrings = []

    def visit_FunctionDef(self, node):
        name = node.name

        # classify
        if name.startswith("__") and name.endswith("__"):
            pass
        elif name.startswith("_"):
            self.non_public_funcs.add(name)
        else:
            self.public_funcs.add(name)

        # add arguments
        for arg in node.args.args:
            if arg.arg not in BUILTIN_NAMES:
                self.non_public_vars.add(arg.arg)

        doc = ast.get_docstring(node)
        if doc:
            self.docstrings.append(
                {
                    "type": "function_docstring",
                    "name": node.name,
                    "line_number": node.lineno,
                    "content": doc,
                }
            )
        self.generic_visit(node)

    def visit_ClassDef(self, node):
        if node.name.startswith("_"):
            self.non_public_classes.add(node.name)
        else:
            self.public_classes.add(node.name)
        self.non_public_vars.add(node.name)

        doc = ast.get_docstring(node)
        if doc:
            self.docstrings.append(
                {
                    "type": "class_docstring",
                    "name": node.name,
                    "line_number": node.lineno,
                    "content": doc,
                }
            )
        self.generic_visit(node)

    def visit_Assign(self, node):
        for target in node.targets:
            if isinstance(target, ast.Name) and target.id not in BUILTIN_NAMES:
                if target.id.isupper():
                    self.public_vars.add(target.id)
                else:
                    self.non_public_vars.add(target.id)
        self.generic_visit(node)

    def visit_Attribute(self, node):
        # Detect method calls like self.get_possible_moves
        if isinstance(node.value, ast.Name) and node.value.id == "self":
            if node.attr not in BUILTIN_NAMES and node.attr not in COMMON_STD_METHODS:
                # treat as a public method
                self.public_funcs.add(node.attr)
        elif node.attr not in BUILTIN_NAMES and node.attr not in COMMON_STD_METHODS:
            # treat other attributes normally
            self.non_public_vars.add(node.attr)
        self.generic_visit(node)

    def visit_Name(self, node):
        if node.id not in BUILTIN_NAMES:
            self.non_public_vars.add(node.id)
        self.generic_visit(node)

    def visit_Module(self, node):
        doc = ast.get_docstring(node)
        if doc:
            self.docstrings.append(
                {
                    "type": "module_docstring",
                    "name": "__main__",
                    "line_number": getattr(node, "lineno", 1),
                    "content": doc,
                }
            )
        self.generic_visit(node)


def get_tokens(code):
    try:
        tree = ast.parse(code)
    except SyntaxError:
        return set()

    visitor = CodeNavigator()
    visitor.visit(tree)

    all_tokens = (
        visitor.public_classes
        # | visitor.public_funcs
        | visitor.non_public_funcs
        | visitor.non_public_vars
        | visitor.non_public_classes
    )

    # 🚫 Remove known stdlib or logging-related names
    cleaned_tokens = {
        t for t in all_tokens if t not in COMMON_STD_METHODS and t not in BUILTIN_NAMES
    }

    return cleaned_tokens


def load_generated_code(file_path):
    if not os.path.exists(file_path):
        return None
    with open(file_path, "r", encoding="utf-8") as f:
        content = f.read()
    return content.strip() if content.strip() else None


#! test generated output
def run_code_with_tests(code, test_imports, tests, return_dict):
    try:
        # Shared environment for both code and tests
        env = {}

        # Run any imports from test_imports
        for imp in test_imports:
            exec(imp, env, env)

        # Execute user code
        exec(code, env, env)

        passed, failed = 0, 0
        return_dict["error"] = ""  # Initialize error as empty string

        # Run all test assertions
        for t in tests:
            try:
                exec(t, env, env)
                passed += 1
            except AssertionError:
                failed += 1
                # print(f"Assertion Error in: {t}")
                return_dict["error"] += f"Assertion Error in: {t}\n"
            except Exception as e:
                failed += 1
                # print(f"Exception Error in: {t} → {e}")
                return_dict["error"] += f"Exception Error in: {t} → {e}\n"

        return_dict["result"] = (passed, failed)

    except Exception as e:
        tb = traceback.format_exc()
        return_dict["error"] = f"Runtime Error in user code:\n{tb}"

    # # Set error to None if no errors occurred
    # if return_dict.get("error", "") == "":
    #     return_dict["error"] = None

    return return_dict


def safe_exec_with_tests(code, test_imports, tests, timeout=2):
    manager = multiprocessing.Manager()
    return_dict = manager.dict()
    p = multiprocessing.Process(
        target=run_code_with_tests, args=(code, test_imports, tests, return_dict)
    )

    p.start()
    p.join(timeout)

    if p.is_alive():
        p.terminate()
        print("⏰ Timeout: possible infinite loop or hanging code.")

        return_dict["result"] = (0, len(tests))
        return_dict["error"] = "Timeout: possible infinite loop or hanging code"
        return return_dict

    if "error" in return_dict:
        return return_dict

    return return_dict


#! Extract starting letters from comments
def extract_comments_from_source(source_code: str) -> List[Dict[str, Any]]:
    comments = []

    # create a deepcopy
    import copy

    source_code = copy.deepcopy(source_code)

    # Split into lines to process each line individually
    lines = source_code.split("\n")

    for line_number, line in enumerate(lines, 1):
        # Find all # symbols and extract comments after them
        hash_index = line.find("#")
        if hash_index != -1:
            # Extract everything after the # symbol
            comment_content = line[hash_index + 1 :].strip()
            if comment_content:  # Skip empty comments
                # Determine if it's an inline comment or full-line comment
                code_before_hash = line[:hash_index].strip()
                comment_type = (
                    "inline_comment" if code_before_hash else "full_line_comment"
                )

                comments.append(
                    {
                        "line_number": line_number,
                        "content": comment_content,
                        "type": comment_type,
                        "code_context": (
                            code_before_hash[:50] + "..."
                            if len(code_before_hash) > 50
                            else code_before_hash
                        ),
                    }
                )
    # Also extract docstrings using AST visitor
    tree = ast.parse(source_code)
    visitor = CodeNavigator()
    visitor.visit(tree)

    docstrings = visitor.docstrings

    comments.extend(docstrings)

    return comments


def get_comment_starting_letters(source_code: str, gamma) -> tuple:

    try:
        comments = extract_comments_from_source(source_code)

        # print(f"EXTRACTED COMMENTS:\n {[comment['content'] for comment in comments]}\n")

        starting_letters = []
        relevant_words = set()

        for comment in comments:
            # Split comment content by newlines to handle multi-line comments
            comment_lines = comment["content"].split("\n")

            for line in comment_lines:
                line = line.strip()
                if not line:
                    continue

                # Extract the first word from this line
                words = re.findall(r"\b[a-zA-Z]+\b", line)
                if words:
                    first_word = words[0].lower()
                    relevant_words.add(first_word)

                    # Get starting letter of the first word
                    if first_word:
                        first_char = first_word[0].lower()
                        if first_char.isalpha():
                            starting_letters.append(first_char)

        print(f"Relevant words: {len(relevant_words)}")

        # Use global green_letters and gamma
        green_count = sum(1 for letter in starting_letters if letter in green_letters)
        token_count = len(starting_letters)

        if token_count > 0:
            p_hat = green_count / token_count
            z_score = (p_hat - gamma) / (
                (gamma * (1 - gamma)) ** 0.5 / token_count**0.5
            )
            p_value = norm.sf(z_score)  # one-tailed test
        else:
            z_score = 0.0
            p_value = 1.0

        return starting_letters, relevant_words, green_count, z_score, p_value

    except Exception as e:
        print(f"❌ Error extracting comment letters: {type(e).__name__}: {e}")
        import traceback

        traceback.print_exc()
        return [], set(), 0, 0.0, 1.0


#! fix the method name with the name mentioned in assert statement


def extract_function_names_from_code(code: str):
    """Extract all function names defined in the user code."""
    tree = ast.parse(code)
    return [node.name for node in ast.walk(tree) if isinstance(node, ast.FunctionDef)]


def extract_function_name_from_tests(test_list):
    """Extract the function name used in assert statements."""
    for test in test_list:
        try:
            tree = ast.parse(test)
            for node in ast.walk(tree):
                # Detect function call inside assert or math.isclose( func(...) )
                if isinstance(node, ast.Call):
                    # Handle nested calls like math.isclose(func(...))
                    for arg in node.args:
                        if isinstance(arg, ast.Call) and isinstance(arg.func, ast.Name):
                            return arg.func.id
                    if isinstance(node.func, ast.Name):
                        return node.func.id
        except SyntaxError:
            continue
    return None


def replace_function_name(code, old_name, new_name):
    """Safely rename the function in the code using AST."""

    class RenameTransformer(ast.NodeTransformer):
        def visit_FunctionDef(self, node):
            if node.name == old_name:
                node.name = new_name
            return node

    tree = ast.parse(code)
    tree = RenameTransformer().visit(tree)
    ast.fix_missing_locations(tree)
    return ast.unparse(tree)


def fix_method_name(user_code, test_list):
    """If needed, rename user's function to match test case function name."""
    code_func_names = extract_function_names_from_code(user_code)
    test_func_name = extract_function_name_from_tests(test_list)

    if not test_func_name:
        print("⚠️ No function found in test cases.")
        return user_code

    # Case 1: If test function name already exists in code, no change needed
    if test_func_name in code_func_names:
        return user_code

    # Case 2: Otherwise, rename the first user function to match test case
    if code_func_names:
        old_name = code_func_names[0]
        updated_code = replace_function_name(user_code, old_name, test_func_name)
        print(f"🔄 Renamed '{old_name}' → '{test_func_name}'")
        return updated_code

    print("⚠️ No function found in user code.")
    return user_code

In [ ]:
import json
from datetime import datetime

FAILURE_LOG_PATH = (
    f"{MODEL_NAME}_failures_{datetime.now().strftime('%Y%m%d_%H_%M_%S')}.json"
)


def log_failure(record_id, failure_type, error_details):
    """Append failure details into a JSON file."""
    log_entry = {
        "task_id": record_id,
        "failure_type": failure_type,
        "error_details": error_details,
        "timestamp": datetime.now().isoformat(),
    }

    try:
        # Load existing log file
        if os.path.exists(FAILURE_LOG_PATH):
            with open(FAILURE_LOG_PATH, "r") as f:
                data = json.load(f)
        else:
            data = []

        data.append(log_entry)

        with open(FAILURE_LOG_PATH, "w") as f:
            json.dump(data, f, indent=4)

    except Exception as e:
        print(f"[Warning] Failed to log error: {e}")

In [ ]:
import math
from scipy.stats import binomtest, norm


def calculate_z_score(token_count, green_count, gamma):
    """Calculate z-score for green token count (standard normal approximation)."""
    if token_count == 0 or gamma <= 0 or gamma >= 1:
        return float("nan")
    return (green_count - gamma * token_count) / math.sqrt(
        gamma * (1 - gamma) * token_count
    )


def calculate_p_value_exact(green_count, token_count, gamma):
    """
    Calculate exact binomial p-value (one-sided test: H1: p > gamma).
    Use this for ALL samples; it's exact and always valid.
    """
    if token_count == 0:
        return float("nan")
    return binomtest(green_count, token_count, gamma, alternative="greater").pvalue


def calculate_p_value_normal(z_score):
    """
    Calculate approximate p-value using standard normal CDF (one-sided).
    Used only for reference/large samples where normal approx is valid.
    """
    if math.isnan(z_score):
        return 1.0
    return norm.sf(z_score)


def get_unified_detection_score(
    token_count, green_count, gamma, small_threshold=SMALL_SAMPLE_THRESHOLD
):
    """
    UNIFIED APPROACH: Compute both z and p_exact, then select based on sample size.
    Returns: (p_unified, z_score, p_exact, method_used)

    For small samples: Use exact binomial p-value (most accurate)
    For large samples: Can use either (both converge), we use p_exact for consistency

    Score for ranking: -log10(p_unified) ensures higher scores = more evidence of watermark
    """
    z_score = calculate_z_score(token_count, green_count, gamma)
    p_exact = calculate_p_value_exact(green_count, token_count, gamma)
    p_normal = calculate_p_value_normal(z_score)

    # Select which p-value to use
    if token_count < small_threshold:
        p_unified = p_exact
        method = "binomial"
    else:
        p_unified = p_exact  # ✅ Use exact for consistency; normal approx also valid
        method = "exact_large"

    # Score for ROC: higher = more evidence. Use -log10 to spread values
    score = -np.log10(np.clip(p_unified, 1e-300, 1.0))

    return {
        "p_unified": p_unified,
        "z_score": z_score,
        "p_exact": p_exact,
        "p_normal": p_normal,
        "score": score,
        "method_used": method,
        "token_count": token_count,
    }

In [ ]:
def detect_watermark(original_code, generated_code, green_letters, red_letters, gamma):
    """
    UNIFIED WATERMARK DETECTION
    - Uses consistent p-value based approach for both decisions and ROC scoring
    - Computes both exact binomial p-value and z-score
    - Decision rule: p_unified < P_THRESHOLD (one-sided)
    - Score for ROC: -log10(p_unified)
    """
    import ast

    def get_starting_letters(code):
        try:
            tree = ast.parse(code)
        except SyntaxError:
            return [], [], 0, float("nan"), float("nan"), {}

        visitor = CodeNavigator()
        visitor.visit(tree)

        # Collect non-public identifiers (user-defined variables, functions, etc.)
        non_public_tokens = (
            visitor.non_public_classes
            | visitor.non_public_funcs
            | visitor.non_public_vars
        )

        # Get starting letters, lowercase, excluding common ones like 'self', 'cls'
        relevant_tokens = [
            token for token in non_public_tokens if token not in {"self", "cls"}
        ]

        def get_starting_letter(word):
            if not word:
                return None
            if word[0] == "_":
                if len(word) > 1 and word[1].isalpha():
                    return word[1].lower()
                else:
                    return None
            elif word[0].isalpha():
                return word[0].lower()
            else:
                return None

        starting_letters = [
            letter
            for word in relevant_tokens
            if (letter := get_starting_letter(word)) is not None
        ]

        green_count = sum(1 for letter in starting_letters if letter in green_letters)

        # Include comment starting letters if enabled
        if COMMENT_ENABLED:
            cmnt_starting_letters, cmn_relevant_words, cmnt_green_count, _, _ = (
                get_comment_starting_letters(code, gamma)
            )
            # print(f"✅ COMMENT STARTING LETTERS: {cmnt_starting_letters}")
            print(
                f"   Comment green count: {cmnt_green_count}, Comment token count: {len(cmnt_starting_letters)}"
            )
            starting_letters.extend(cmnt_starting_letters)
            relevant_tokens.extend(cmn_relevant_words)
            green_count += cmnt_green_count

        token_count = len(starting_letters)

        # ✅ Get unified detection stats
        unified_stats = get_unified_detection_score(token_count, green_count, gamma)

        return starting_letters, relevant_tokens, green_count, unified_stats

    orig_starts, orig_relevant_tokens, orig_green_count, orig_stats = (
        get_starting_letters(original_code)
    )
    # print("ORIGINAL TOKENS: ", orig_relevant_tokens, "LEN: ", len(orig_relevant_tokens))
    # print("ORIGINAL GREEN COUNT: ", orig_green_count)
    # print(
    #     f"ORIGINAL STATS: method={orig_stats['method_used']}, p={orig_stats['p_unified']:.6f}, z={orig_stats['z_score']:.4f}"
    # )

    gen_starts, gen_relevant_tokens, gen_green_count, gen_stats = get_starting_letters(
        generated_code
    )
    # print("GENERATED TOKENS: ", gen_relevant_tokens, "LEN: ", len(gen_relevant_tokens))
    # print("GENERATED GREEN COUNT: ", gen_green_count)
    # print(
    #     f"GENERATED STATS: method={gen_stats['method_used']}, p={gen_stats['p_unified']:.6f}, z={gen_stats['z_score']:.4f}"
    # )

    preferred = green_letters
    avoided = red_letters

    # Calculate ratios
    orig_preferred_ratio = (
        sum(1 for letter in orig_starts if letter in preferred) / len(orig_starts)
        if orig_starts
        else 0
    )
    gen_preferred_ratio = (
        sum(1 for letter in gen_starts if letter in preferred) / len(gen_starts)
        if gen_starts
        else 0
    )

    orig_avoided_count = sum(1 for letter in orig_starts if letter in avoided)
    gen_avoided_count = sum(1 for letter in gen_starts if letter in avoided)

    # ✅ UNIFIED DETECTION: Use p_unified for both decision and all returned stats
    def is_watermarked_unified(p_unified, threshold=P_THRESHOLD):
        """Simple threshold-based decision using unified p-value."""
        if math.isnan(p_unified):
            return False
        return p_unified < threshold

    return {
        # Original code stats
        "original_z_score": orig_stats["z_score"],
        "original_p_exact": orig_stats["p_exact"],
        "original_p_unified": orig_stats["p_unified"],
        "original_score": orig_stats["score"],
        "original_method_used": orig_stats["method_used"],
        "original_preferred_ratio": orig_preferred_ratio,
        "original_token_count": len(orig_relevant_tokens),
        "original_green_count": orig_green_count,
        "original_is_watermarked": is_watermarked_unified(orig_stats["p_unified"]),
        "original_unique_starts": sorted(set(orig_starts)),
        # Generated code stats
        "generated_z_score": gen_stats["z_score"],
        "generated_p_exact": gen_stats["p_exact"],
        "generated_p_unified": gen_stats["p_unified"],
        "generated_score": gen_stats["score"],
        "generated_method_used": gen_stats["method_used"],
        "generated_preferred_ratio": gen_preferred_ratio,
        "generated_token_count": len(gen_relevant_tokens),
        "generated_green_count": gen_green_count,
        "generated_is_watermarked": is_watermarked_unified(gen_stats["p_unified"]),
        "generated_unique_starts": sorted(set(gen_starts)),
    }

### Watermark Embedding


#### Gemini-2.5-Flash


In [ ]:
from google import genai
from google.genai import types
import os
import re
from pathlib import Path
import ast

API_KEY = os.getenv("GEMINI_API_KEY") or os.getenv("API_KEY")

if not API_KEY:
    raise RuntimeError(
        "Missing API key. Please set GEMINI_API_KEY or API_KEY in your environment (e.g., .env)."
    )

client = genai.Client(api_key=API_KEY)

#### PROMPT TEMPLATE REFERENCE:

````tex
Wang, C. Y., DaghighFarsoodeh, A., & Pham, H. V. (2024).
Selection of prompt engineering techniques for code generation through predicting code complexity.
arXiv preprint arXiv:2409.16416.```
````


In [ ]:
natural_token_examples = {
  "a": [
    "arr",
    "arr1",
    "angle",
    "average",
    "add",
    "area",
    "all_numbers",
    "angle_in_radians"
  ],
  "b": ["base1", "base2", "base", "bit", "bisect"],
  "c": [
    "ch",
    "char",
    "current_number",
    "current_tuple",
    "current_position",
    "current_list",
    "center_point",
    "current_char",
    "cost_price",
    "current_length",
    "check_element",
    "count_Substrings",
    "common_elements"
  ],
  "d": [
    "dp",
    "digits",
    "dictionary",
    "divisor",
    "digit",
    "dict1",
    "dict2",
    "drop_empty",
    "diff",
    "difference"
  ],
  "e": [
    "ele",
    "element",
    "extract_nth_element",
    "expected_num",
    "effective_rotations",
    "even_numbers",
    "even_count",
    "extracted_elements",
    "elements",
    "even_number",
    "extracted_strings"
  ],
  "f": [
    "first_even",
    "first_odd",
    "first_index",
    "final_output",
    "find_even_pair",
    "first_elements",
    "final_result",
    "first_collection",
    "first_pos",
    "first_tuple"
  ],
  "g": [
    "given_list",
    "grouped_collections",
    "grade_list",
    "groups",
    "gcd_value",
    "group_part",
    "get_char_count_array",
    "grid_position"
  ],
  "h": [
    "high",
    "height",
    "has_vowel_neighbor",
    "hypotenuse_length",
    "hypotenuse_squared",
    "half_base_edge",
    "has_31_days",
    "has_lowercase_underscore_sequence"
  ],
  "i": [
    "idx",
    "input_list",
    "item",
    "i",
    "islower",
    "isupper",
    "isdigit",
    "is_sublist",
    "input_tuple",
    "inner_tuple"
  ],
  "j": ["joined_result"],
  "k": ["known_length", "key"],
  "l": [
    "lst",
    "list_of_lists",
    "length",
    "low",
    "list1",
    "last_index",
    "list2",
    "list_of_tuples",
    "list_length",
    "lists"
  ],
  "m": [
    "mid",
    "math",
    "max_length",
    "main_list",
    "main_string",
    "max_prod",
    "min_prod",
    "matches",
    "max_size",
    "merged_dict"
  ],
  "n": [
    "numbers",
    "num",
    "number",
    "numbers_list",
    "NO_OF_CHARS",
    "nested_list",
    "number_list",
    "num1",
    "normalized_pair"
  ],
  "o": [
    "original_list",
    "origin_position",
    "other_tuple",
    "odd_count",
    "occurrence_tracker",
    "old_char",
    "occurrences",
    "odd_num_sum",
    "overall_max_product",
    "occurrence_frequency"
  ],
  "p": [
    "position",
    "pair",
    "product",
    "pattern",
    "pi",
    "power",
    "position_counter",
    "products",
    "previous",
    "positions"
  ],
  "q": [
    "qualified_terms",
    "qurrent_length",
    "quickest_record",
    "quotient",
    "qualified_count",
    "quoted_portions",
    "quadratic_results",
    "qualified_numbers",
    "quoted_text"
  ],
  "r": [
    "result",
    "result_list",
    "rotation_count",
    "ranges",
    "rotations",
    "rounded_numbers",
    "right",
    "right_pointer",
    "radius",
    "remaining_part"
  ],
  "s": [
    "sublist",
    "str1",
    "s1",
    "start",
    "sqrt",
    "start_index",
    "s2",
    "seq",
    "str_digits",
    "sorted_list"
  ],
  "t": [
    "temp",
    "total_sum",
    "text",
    "tuple_list",
    "text_string",
    "target",
    "total",
    "text1",
    "tuple_pair",
    "tuples"
  ],
  "u": [
    "unique_numbers",
    "unique_pairs",
    "unique_values",
    "unique_elements",
    "unknown_leg_squared",
    "union_elements",
    "unique_product"
  ],
  "v": ["value", "vowels", "value_set"],
  "w": ["word", "words", "width", "words_list"],
  "x": ["x1", "x_prime"],
  "y": ["y1", "y_prime"],
  "z": ["zeros_to_ones"]
}
# natural_token_examples

In [ ]:
SYSTEM_PROMPT = """
## Additional Instruction:
#### Green Letter List: {green_words}
#### Green Letter Examples: {green_examples}
#### Red Letter List: {red_words}

### Command:
Generate code following the given instructions:
1. Green Letter Enriched Identifier: When generating identifiers (for example: local variables, function parameters, private helper functions, internal class attributes, temporary variables) & comments, start the words with letters from the 'Green Letter List'. Use the provided examples as reference for naming patterns.
2. Correct & Relevant: Generate correct code following the problem statement.
3. About comments: Add brief comments only to clarify complex logic, but do not over-comment or exaggerate. Avoid tokens that starts with the Red List letters.
4. About Method Name: Write the method name as mentioned in the given test case.
5. Warning: Do not mention or explain the renaming rules in your output.
6. Others: Provide explanation within 3 bullet points, justifying why your solution is correct outside the code block & at the end of the response. If generated code fails tests, then follow the explanation and write down the correct code. Never include the given test cases, explanation or assertions inside the codeblocks.

### Identifier Naming Rules:
- Always choose semantically relevant names that describe the variables purpose. 
- If multiple green-list tokens are available, prefer the one that forms a meaningful compound name (e.g., `total_sum`, `user_count`) rather than arbitrary forms like `a_value`.
- Avoid random short tokens (a_, x_, ab_) like a_value, x_unit, c_val_unused.
- Prefer human-like identifiers similar to total_cost, user_count, num_records.
- Use short token only when necessary for loop indices or temporary variables like i, j, temp.
- Reference the provided green letter examples for naming inspiration.
"""

PROBLEM_TEMPLATE = (
    "You are a helpful code assistant that can teach a junior developer how to code."
    "Your language of choice is Python. Generate the Python code for the following task enclosed in ```python```\n\n"
    "##Prompt:\n{prompt}\n\n"
    "##Test Cases:\n{tests}\n\n"
)

In [ ]:
def generate_code(record, feedback_prompt=""):

    #! Get green and red letter sets
    task_id = record["task_id"]
    global green_letters, red_letters
    green_letters, red_letters, ratio = get_red_green_sets(task_id)

    print(
        f"✅ Generated green/red letter sets for task {task_id} with ratio {ratio:.3f}"
    )
    print("GREEN LETTERS:", green_letters)
    print("RED LETTERS:", red_letters)

    # Build green letter examples from natural_token_examples
    green_examples = {l: natural_token_examples[l][:5] for l in green_letters if l in natural_token_examples.keys()}
    green_examples_str = ", ".join([f"{letter}: [{', '.join(examples)}]" for letter, examples in sorted(green_examples.items())])

    problem_query = record["prompt"]
    tests = "\n".join(record["test_list"])
    system_instruction = SYSTEM_PROMPT.format(
        green_words=green_letters, green_examples=green_examples_str, red_words=red_letters
    )
    full_prompt = PROBLEM_TEMPLATE.format(prompt=problem_query, tests=tests)

    if len(feedback_prompt) > 0:
        full_prompt = full_prompt + "\n\n" + feedback_prompt

    # print("\n>> FULL PROMPT: ", system_instruction, "\n")

    contents = [
        types.Content(
            role="user",
            parts=[types.Part.from_text(text=full_prompt)],
        )
    ]

    response = client.models.generate_content(
        model=MODEL_PATH,
        contents=contents,
        config=types.GenerateContentConfig(system_instruction=system_instruction),
    )

    full_text = response.text.strip()  # full reasoning + code
    code_blocks = re.findall(r"```python(.*?)```", full_text, re.DOTALL)
    actual_code_blocks = [block.strip() for block in code_blocks if block.strip()]

    # Extract code & reasoning separately
    code_text = actual_code_blocks[-1] if actual_code_blocks else ""
    explanation_text = re.sub(
        r"```python.*?```", "", full_text, flags=re.DOTALL
    ).strip()

    return {
        "code": code_text,
        "explanation": explanation_text,
        "full_response": full_text,
    }

### Execution


In [ ]:
PHASE1_DEFAULT_Z_THRESHOLD = Z_THRESHOLD  # sensible default if not supplied


def _get_tests_from_record(record):
    """Return (test_imports, tests_list) from a record safely."""

    test_imports = record.get("test_imports", []) or record.get("imports", []) or []
    tests_list = record.get("test_list", []) or record.get("tests", []) or []

    # Normalize to list of strings
    if isinstance(test_imports, str):
        test_imports = [test_imports]
    if isinstance(tests_list, str):
        tests_list = [tests_list]
    return test_imports, tests_list


def evaluate_candidate(record, code, z_threshold=PHASE1_DEFAULT_Z_THRESHOLD):
    """
    Compute (correctness_bool, passed, failed, z, p_unified, p_exact, score, method_used, token_count, green_count, meets_z).
    ✅ UNIFIED APPROACH: Uses exact binomial test for all samples (consistent with fixed detection method).
    Uses existing helpers: run_code_with_tests, detect_watermark.
    """

    #! Get green/red letter sets for this task
    task_id = record["task_id"]

    letter_freqs = freq_data['letter_freqs']
    total_identifiers = freq_data['total_identifiers']

    global green_letters, red_letters
    green_letters, red_letters, ratio = get_red_green_sets(task_id)
    GAMMA = calculate_gamma(letter_freqs, total_identifiers, green_letters)

    print(f"✅ Evaluating candidate for task {task_id} with ratio {ratio}")
    print("GREEN LETTERS:", green_letters)
    print("GAMMA:", GAMMA)
    print("CANDIDATE CODE:\n", code, "\n")

    # Correctness via unit tests
    test_imports, tests_list = _get_tests_from_record(record)
    test_result = safe_exec_with_tests(code, test_imports, tests_list)

    passed, failed = test_result["result"]
    error_message = test_result["error"]
    correctness = (failed == 0) and (passed > 0)

    # Watermark fidelity via unified detection (exact binomial + z-score hybrid)
    original_code = record.get("code", "") or ""
    detection_result = (
        detect_watermark(original_code, code, green_letters, red_letters, GAMMA)
        if code
        else {}
    )

    # ✅ FIXED: Extract all necessary fields with CORRECT keys from detect_watermark return dict
    z = detection_result.get("generated_z_score", None)
    p_unified = detection_result.get(
        "generated_p_unified", None
    )  # ✅ CORRECT KEY (was generated_p_value)
    p_exact = detection_result.get(
        "generated_p_exact", None
    )  # ✅ NEW: Exact binomial p-value
    score = detection_result.get(
        "generated_score", float("nan")
    )  # ✅ NEW: For ROC analysis
    method_used = detection_result.get(
        "generated_method_used", "unknown"
    )  # ✅ NEW: Track method used

    token_count = detection_result.get("generated_token_count", 0)
    green_count = detection_result.get("generated_green_count", 0)
    green_tokens = detection_result.get("generated_unique_starts", [])

    meets_z = bool(detection_result.get("generated_is_watermarked", False))

    return {
        "correctness": bool(correctness),
        "error_message": error_message if len(error_message) > 0 else None,
        "passed": int(passed),
        "failed": int(failed),
        "z": z,
        "p_unified": p_unified,  # ✅ NEW: Unified p-value
        "p_exact": p_exact,  # ✅ NEW: Exact binomial p-value
        "score": score,  # ✅ NEW: Watermark evidence score for ROC
        "method_used": method_used,  # ✅ NEW: Track which method was used
        "token_count": int(token_count) if isinstance(token_count, (int, float)) else 0,
        "green_count": int(green_count) if isinstance(green_count, (int, float)) else 0,
        "meets_z": bool(meets_z),
        "watermark_failed": not meets_z,
        "watermark_details": (
            (z, p_unified, token_count, green_count, green_tokens)
            if not meets_z
            else None
        ),
    }


def run_phase1(
    record, z_threshold=PHASE1_DEFAULT_Z_THRESHOLD, iter_cap=3, verbose=True
):
    best = None
    selected = None
    feedback_prompt = ""

    for it in range(1, int(iter_cap) + 1):
        if verbose:
            print(f"[{record['task_id']}] Iteration {it} / {iter_cap}")

        gen = generate_code(record, feedback_prompt)
        code = gen["code"]
        reasoning_text = gen["explanation"]

        eval_res = evaluate_candidate(record, code, z_threshold=z_threshold)
        eval_res.update(
            {
                "iteration": it,
                "reasoning_text": reasoning_text,
                "full_llm_response": gen["full_response"],
            }
        )

        eval_res["stopping_condition_met"] = (
            eval_res["correctness"] and eval_res["meets_z"]
        )
        eval_res["code"] = code

        # if verbose:
        #     print(f"EVAL RESULT: ", eval_res)

        # --- Save reasoning if failure ---
        if not eval_res["stopping_condition_met"]:
            failure_entry = {
                "task_id": record["task_id"],
                "iteration": it,
                "timestamp": datetime.now().isoformat(),
                "correctness": bool(eval_res["correctness"]),
                "meets_z": bool(eval_res["meets_z"]),
                "error_message": eval_res.get("error_message"),
                "llm_reasoning": reasoning_text,
                "z": eval_res.get("z"),
                "p_unified": eval_res.get("p_unified"),  # ✅ NEW
                "p_exact": eval_res.get("p_exact"),  # ✅ NEW
                "score": eval_res.get("score"),  # ✅ NEW
                "method_used": eval_res.get("method_used"),  # ✅ NEW
            }
            with open(FAILURE_LOG_PATH, "a") as f:
                f.write(json.dumps(failure_entry) + "\n")

        # Early stopping
        if eval_res["stopping_condition_met"]:
            selected = eval_res
            if verbose:
                p_val = eval_res.get("p_unified", float("nan"))
                print(
                    f"[{record['task_id']}] ✅ Stop: Correct & p_unified={p_val:.6f} < {P_THRESHOLD}"
                )
            break

        if not eval_res["correctness"]:
            error_log = eval_res.get("error_message")
            feedback_prompt = f"❌ The previous code execution failed.\nAnalyze the error step by step:\n1. Review the error log: {error_log}\n2. Consider the previous reasoning: {reasoning_text}\n\nNow write the explanation why it failed. \nThen generate the corrected version of the code that fixes the issues. \nEnsure the code is executable following the problem requirements."
        elif eval_res["watermark_failed"]:
            z, p_unified, token_count, green_count, green_tokens = eval_res[
                "watermark_details"
            ]
            score = eval_res.get("score", float("nan"))
            feedback_prompt = f"Your provided code is current but watermark fidelity failed (p-value={p_unified:.6f}, P_THRESHOLD={P_THRESHOLD:.6f}).\nGreen Token Count={green_count}/{token_count}, Green Letters={green_tokens}.\nUpdate the code to use more green-letter identifiers (e.g., {', '.join(list(green_letters)[:5])}) in identifiers, and comments. Maintain functional correctness.\n"

        # Update best result (using unified p-value score for ranking: higher score = stronger watermark)
        current_score = eval_res.get("score", -1e9)
        if math.isnan(current_score):
            current_score = -1e9

        best_score = best.get("score", -1e9) if best else -1e9
        if math.isnan(best_score):
            best_score = -1e9

        if best is None or (
            (eval_res["correctness"] and not best["correctness"])
            or (current_score > best_score)
        ):
            best = eval_res

    if selected is None:
        selected = best

    return selected or {}

In [ ]:
import time

def process_dataset(df, output_dir):
    Path(output_dir).parent.mkdir(exist_ok=True)
    results = []

    for idx, record in df.iterrows():
        task_id = record.get("task_id") or record.get("id")
        out_dir = Path(output_dir)
        out_dir.mkdir(parents=True, exist_ok=True)

        try:
            sel = run_phase1(record, z_threshold=Z_THRESHOLD, iter_cap=ITER_CAP, verbose=True)
            code = sel.get("code", "") if sel else ""
        
            # Save code
            output_file = out_dir / f"{task_id}.py"
            with open(output_file, "w", encoding="utf-8") as f:
                f.write(code or "")

            # Persist result row
            results.append(
                {
                    "task_id": task_id,
                    "iteration_used": sel.get("iteration"),
                    "correct": sel.get("correctness"),
                    "tests_passed": sel.get("passed"),
                    "tests_failed": sel.get("failed"),
                    "z_score": sel.get("z"),
                    "p_unified": sel.get("p_unified"),             # ✅ NEW: Unified p-value
                    "p_exact": sel.get("p_exact"),                 # ✅ NEW: Exact binomial p-value
                    "score": sel.get("score"),                     # ✅ NEW: Watermark evidence score
                    "method_used": sel.get("method_used"),         # ✅ NEW: Detection method used
                    "token_count": sel.get("token_count"),
                    "green_count": sel.get("green_count"),
                    "meets_z": sel.get("meets_z"),
                    "stopped_early": sel.get("stopping_condition_met"),
                    "output_file": str(output_file),
                }
            )
        except Exception as e:
            print(f"[Phase1] ❌ Failure for {task_id}: {e}")
            results.append(
                {
                    "task_id": task_id,
                    "error": str(e),
                    "iteration_used": None,
                    "correct": False,
                    "tests_passed": 0,
                    "tests_failed": 0,
                    "z_score": float("nan"),
                    "p_unified": float("nan"),                     # ✅ NEW
                    "p_exact": float("nan"),                       # ✅ NEW
                    "score": float("nan"),                         # ✅ NEW
                    "method_used": "error",                        # ✅ NEW
                    "token_count": 0,
                    "green_count": 0,
                    "meets_z": False,
                    "stopped_early": False,
                    "output_file": None,
                }
            )

        print(f"\n{idx+1}/{len(df)} processed: {task_id}\n")

    results_df = pd.DataFrame(results)
    Path(RESULTS_CSV).parent.mkdir(parents=True, exist_ok=True)

    results_df.to_csv(RESULTS_CSV, index=False)
    print(f"Results saved to {RESULTS_CSV}")

    return results

### RUN

In [ ]:
results = process_dataset(df, OUTPUT_DIR)